In [1]:
pip install torch torchvision datasets pillow numpy opencv-python scikit-learn

In [2]:
from datasets import load_dataset

import numpy as np
from sklearn.model_selection import train_test_split

import random
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torch.nn as nn

In [3]:
# Loads the Fives dataset from Hugging Face
print("Loading dataset from Hugging Face")
dataset = load_dataset("tyluan/FIVES")
print(dataset)
print("Dataset Loaded")

Loading dataset from Hugging Face


README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  453MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  395MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

... (3 outputs left out)

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'image_id'],
        num_rows: 600
    })
    test: Dataset({
        features: ['image', 'mask', 'image_id'],
        num_rows: 200
    })
})
Dataset Loaded


In [4]:
# Splits the dataset into training, validation and testing
# Training images - 480
# Validatoin images - 120
# Testing images - 200
full_training_dataset = dataset["train"]

training_images, validation_images = train_test_split(
    np.arange(len(full_training_dataset)),
    test_size=0.2,
    random_state=17,
    shuffle=True,
    )

training_dataset_split = full_training_dataset.select(training_images)
validation_dataset_split = full_training_dataset.select(validation_images)
testing_dataset_split = dataset["test"]

print("Training dataset size: ", len(training_dataset_split))
print("Validation dataset size: ", len(validation_dataset_split))
print("Testing dataset size: ", len(testing_dataset_split))

Training dataset size:  480
Validation dataset size:  120
Testing dataset size:  200


In [5]:
# Formating the dataset so it fits our model
class FIVESDataset(Dataset):
  def __init__(self, data, target_size=(512,512), is_training=False):
    self.data = data
    self.target_size = target_size
    self.is_training = is_training

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    image_mask_pair = self.data[index]

    image = image_mask_pair["image"]
    mask = image_mask_pair["mask"]

    image = TF.resize(image, self.target_size, interpolation=TF.InterpolationMode.BILINEAR)
    mask = TF.resize(mask, self.target_size, interpolation=TF.InterpolationMode.NEAREST)

    # Randomly augment some of the training dataset
    if self.is_training:
      if random.random() < 0.5:
        image = TF.hflip(image)
        mask = TF.hflip(mask)
      if random.random() < 0.5:
        image = TF.vflip(image)
        mask = TF.vflip(mask)

    image = TF.to_tensor(image)
    mask = TF.to_tensor(mask)
    mask = (mask > 0.5).float()

    return image, mask

In [6]:
# Instantiate the three dataset splits
training_dataset = FIVESDataset(training_dataset_split, target_size=(512, 512), is_training=True)
validation_dataset = FIVESDataset(validation_dataset_split, target_size=(512, 512), is_training=False)
testing_dataset = FIVESDataset(testing_dataset_split, target_size=(512, 512), is_training=False)

# Image shape - [3, 512, 512]
# Mask shape - [1, 512, 512]
image, mask = training_dataset[random.randint(0, len(training_dataset))]
print("Image shape: ", image.shape)
print("Mask shape: ", mask.shape)

Image shape:  torch.Size([3, 512, 512])
Mask shape:  torch.Size([1, 512, 512])


In [7]:
# Load data for training
batch_size = 4

training_loader = DataLoader(training_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(testing_dataset, batch_size=batch_size, shuffle=False)

batch_images, batch_masks = next(iter(training_loader))

print("Batch check:")
print(f"  Batch images shape: {batch_images.shape} (Batch, Channel, Height, Width)")
print(f"  Batch masks shape:  {batch_masks.shape}  (Batch, Channel, Height, Width)")
print(f"  Total train batches: {len(training_loader)} ({len(training_dataset)} / {batch_size})")
print(f"  Total val batches:   {len(validation_loader)}   ({len(validation_dataset)} / {batch_size})")
print(f"  Total test batches:  {len(test_loader)}  ({len(testing_dataset)} / {batch_size})")

Batch check:
  Batch images shape: torch.Size([4, 3, 512, 512]) (Batch, Channel, Height, Width)
  Batch masks shape:  torch.Size([4, 1, 512, 512])  (Batch, Channel, Height, Width)
  Total train batches: 120 (480 / 4)
  Total val batches:   30   (120 / 4)
  Total test batches:  50  (200 / 4)


In [15]:
# Standard UNET implementation
def convolution_block(in_channels, out_channels):
  return nn.Sequential(
      nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
      nn.BatchNorm2d(out_channels),
      nn.ReLU(inplace=True),
      nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
      nn.BatchNorm2d(out_channels),
      nn.ReLU(inplace=True)
  )


class UNET(nn.Module):
  def __init__(self, in_channels=3, out_channels=1):
    super().__init__()
    self.pool = nn.MaxPool2d(2)

    # 1. Encoder (Downsampling path)
    self.enc1 = convolution_block(in_channels, 64)
    self.enc2 = convolution_block(64, 128)
    self.enc3 = convolution_block(128, 256)
    self.enc4 = convolution_block(256, 512)

    # 2. Bottleneck
    self.bottleneck = convolution_block(512, 1024)

    # 3. Decoder (Upsampling path + Convolutions)
    self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
    self.dec4 = convolution_block(1024, 512)   # 512 (up4) + 512 (enc4) = 1024 in

    self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
    self.dec3 = convolution_block(512, 256)    # 256 (up3) + 256 (enc3) = 512 in

    self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
    self.dec2 = convolution_block(256, 128)    # 128 (up2) + 128 (enc2) = 256 in

    self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
    self.dec1 = convolution_block(128, 64)     # 64 (up1) + 64 (enc1) = 128 in

    # 4. Final 1x1 projection to raw logits
    self.out = nn.Conv2d(64, out_channels, kernel_size=1)

  def forward(self, x):
      # Encoder passes
      e1 = self.enc1(x)
      e2 = self.enc2(self.pool(e1))
      e3 = self.enc3(self.pool(e2))
      e4 = self.enc4(self.pool(e3))

      # Bottleneck
      b = self.bottleneck(self.pool(e4))

      # Decoder passes with skip connections
      d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
      d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
      d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
      d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

      return self.out(d1)
